In [5]:
# %% [markdown]
# # 🎛️ CONTROL CENTRAL: DASHBOARD INTERACTIVO DE INGENIERÍA
# #### Celda 1: Importación de librerías, carga de datos y limpieza total de la base.

# %%
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carga segura de tu archivo consolidado (2020-2025)
path_csv = 'csv/BaseINDICES-2020-2025.csv'
try:
    df_base = pd.read_csv(path_csv, sep=';', encoding='utf-8')
except Exception:
    df_base = pd.read_csv(path_csv, sep=',', encoding='utf-8')

# 2. Limpieza estricta de las columnas numéricas para evitar errores de tipo string
cols_numericas = ['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Vacantes', 'Matrícula Primer Año', 'Valor de arancel', 'Matrícula Total']
for col in cols_numericas:
    if col in df_base.columns and df_base[col].dtype == 'object':
        df_base[col] = df_base[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    df_base[col] = pd.to_numeric(df_base[col], errors='coerce')

df_base['Año'] = pd.to_numeric(df_base['Año'], errors='coerce').fillna(2024).astype(int)

# 3. Filtro macro: Tratamos todo bajo el gran bloque unificado de Ingeniería
df_ing = df_base[df_base['Carrera Genérica'].str.contains('Ingeniería', case=False, na=False)].copy()

# 4. Mapeo automático de columnas de la base de datos (CNED / SIES)
col_inst = 'Institución' if 'Institución' in df_ing.columns else ('Nombre Institución' if 'Nombre Institución' in df_ing.columns else 'Institución')
col_reg = 'Nombre Región' if 'Nombre Región' in df_ing.columns else ('Región' if 'Región' in df_ing.columns else 'Sede')

# Extraemos las opciones reales directamente desde el archivo para evitar descalces
lista_instituciones = ['---'] + sorted(df_ing[col_inst].dropna().unique().tolist()) if col_inst in df_ing.columns else ['---', 'Universidad Austral de Chile']
lista_carreras = ['---'] + sorted(df_ing['Carrera Genérica'].dropna().unique().tolist())

print(f"✅ BASE DE DATOS OPTIMIZADA: {len(df_ing)} filas listas para el motor reactivo.")

✅ BASE DE DATOS OPTIMIZADA: 75 filas listas para el motor reactivo.


In [6]:
# %% [markdown]
# #### Celda 2: Maquetación y Arquitectura Visual de la Interfaz (Widgets y HTML)

# %%
# 1. Estilos CSS para los bordes redondeados de la barra de navegación
estilos_css = widgets.HTML("""
<style>
    .tabs-redondeadas .btn {
        border-radius: 16px !important; 
        margin-right: 6px !important;   
        border: 1px solid #bce8f1 !important;
    }
</style>
""")

# 2. Control de navegación superior
tabs_navegacion = widgets.ToggleButtons(
    options=['KPIs', 'GRÁFICOS', 'MAPAS', 'ML (RANDOM FOREST PESOS)'],
    value='KPIs',
    button_style='info',
    layout=widgets.Layout(width='100%', margin='0px 0px 5px 0px')
)
tabs_navegacion.add_class('tabs-redondeadas') 

linea_separadora = widgets.HTML("<hr style='border: 0; border-top: 3px solid #000000; margin: 5px 0px 15px 0px; width: 100%; opacity: 1;'>")

# =====================================================================
# INTERFAZ INTERNA: PESTAÑA 'KPIs'
# =====================================================================
selector_kpi_institucion = widgets.Dropdown(options=lista_instituciones, value='---', description='Institución:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
selector_kpi_carrera = widgets.Dropdown(options=lista_carreras, value='---', description='Carrera:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='5px 0px 15px 0px'))

panel_izquierdo_kpis = widgets.VBox([
    widgets.HTML("<h4>Filtros KPI</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_kpi_institucion,
    selector_kpi_carrera,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'><i>Selecciona una institución y carrera para calcular indicadores en tiempo real desde la BD.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_kpi_cards = widgets.Output(layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px'))
layout_tab_kpis = widgets.HBox([panel_izquierdo_kpis, area_kpi_cards], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

# =====================================================================
# INTERFAZ INTERNA: PESTAÑA 'GRÁFICOS'
# =====================================================================
selector_graficos = widgets.Dropdown(options=['---', 'Evolución de Arancel Promedio', 'Evolución de Matrícula Total', 'Tendencia de Ingreso Femenino'], value='---', description='Gráfico:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))
selector_dimensiones = widgets.Dropdown(options=lista_carreras, value='---', description='Carrera:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))

panel_izquierdo_graficos = widgets.VBox([
    widgets.HTML("<h4>Reportes Estadísticos</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_graficos,
    selector_dimensiones,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'><i>Genera curvas de tendencias completas (2020-2025) aplicando filtros interactivos.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_grafico = widgets.Output(layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px'))
layout_tab_graficos = widgets.HBox([panel_izquierdo_graficos, area_imagen_grafico], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

# =====================================================================
# INTERFAZ INTERNA: PESTAÑA 'MAPAS'
# =====================================================================
selector_mapas = widgets.Dropdown(options=['---', 'Distribución de Alumnos por Región/Sede', 'Comparativa de Aranceles Regiones'], value='---', description='Ver Reporte:', style={'description_width': 'initial'}, layout=widgets.Layout(width='95%', margin='10px 0px'))

panel_izquierdo_mapas = widgets.VBox([
    widgets.HTML("<h4>Variables Territoriales</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_mapas,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'><i>Despliega el comportamiento analítico segmentado geográficamente.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_mapa = widgets.Output(layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px'))
layout_tab_mapas = widgets.HBox([panel_izquierdo_mapas, area_imagen_mapa], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

# Lienzo para algoritmos predictivos
layout_tab_ml = widgets.VBox([
    widgets.HTML("<div style='padding: 40px; text-align: center; color: #999; font-style: italic; font-family: sans-serif;'><h3>[ Pestaña de ML Limpia ]</h3>Espacio listo para integrar tu Random Forest.</div>")
], layout=widgets.Layout(width='100%', height='100%'))

contenedor_cuerpo = widgets.Output(layout=widgets.Layout(width='100%', height='430px', overflow='auto'))
sns.set_theme(style="whitegrid")
print("✅ CELDA 2 CONTINUA: Maquetación y layouts listos.")

✅ CELDA 2 CONTINUA: Maquetación y layouts listos.


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Interfaz Interactiva del Dashboard            ║
# ╚══════════════════════════════════════════════════════════╝
DATOS = {
    "Ingenierías UACh (2020-2025)": df_ing
}
# ── Panel de selección de archivo ───────────────────────────────────────────
selector_archivo = widgets.Dropdown(
    options={nombre: nombre for nombre in sorted(DATOS.keys())},
    description='Archivo:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de eje X ─────────────────────────────────────────────
selector_x = widgets.Dropdown(
    options=['Seleccionar...'],
    description='Eje X:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de eje Y ─────────────────────────────────────────────
selector_y = widgets.Dropdown(
    options=['Seleccionar...'],
    description='Eje Y:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de tipo de gráfico ───────────────────────────────────
selector_tipo = widgets.Dropdown(
    options=['auto', 'barras', 'linea', 'histograma', 'dispersion', 'caja'],
    value='auto',
    description='Tipo:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de color ─────────────────────────────────────────────
selector_color = widgets.Dropdown(
    options=['—'],
    value='—',
    description='Color por:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Botón para generar gráfico ──────────────────────────────────────────────
btn_generar = widgets.Button(
    description='📊 Generar Gráfico',
    button_style='info',
    tooltip='Genera el gráfico con los parámetros seleccionados',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Botón para actualizar estadísticas ──────────────────────────────────────
btn_estadisticas = widgets.Button(
    description='📈 Estadísticas',
    button_style='success',
    tooltip='Muestra resumen estadístico del archivo',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Botón para recargar archivos ────────────────────────────────────────────
btn_recargar = widgets.Button(
    description='🔄 Recargar',
    button_style='warning',
    tooltip='Recarga los archivos CSV de la carpeta',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Área de salida para gráficos ────────────────────────────────────────────
output_grafico = widgets.Output()

# ── Área de salida para estadísticas ────────────────────────────────────────
output_estadisticas = widgets.Output()

# ── Mensaje de estado ───────────────────────────────────────────────────────
label_estado = widgets.HTML(value="<b style='color:#666;'>⏳ Listo para generar gráficos</b>")


# ── Función para recargar archivos CSV ──────────────────────────────────────
def recargar_archivos(button):
    """Recarga dinámicamente los CSV de la carpeta."""
    global DATOS
    with output_grafico:
        output_grafico.clear_output(wait=True)
        label_estado.value = "<b style='color:#EA580C;'>⏳ Recargando archivos...</b>"
        
        try:
            DATOS.clear()
            DATOS = cargar_todos_los_csv(CSV_DIR)
            
            # Actualizar opciones del selector
            nuevos_archivos = sorted(DATOS.keys())
            selector_archivo.options = {n: n for n in nuevos_archivos}
            
            if nuevos_archivos:
                selector_archivo.value = nuevos_archivos[0]
                actualizar_columnas({'new': nuevos_archivos[0]})
                label_estado.value = f"<b style='color:#0D9488;'>✓ {len(DATOS)} archivo(s) cargado(s)</b>"
            else:
                label_estado.value = "<b style='color:#EA580C;'>⚠️  No se encontraron archivos CSV</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error al recargar: {str(e)}</b>"


# ── Función para actualizar opciones de columnas ────────────────────────────
def actualizar_columnas(change):
    """Actualiza las opciones de X, Y y Color cuando cambia el archivo."""
    archivo = change['new']
    if archivo in DATOS:
        df = DATOS[archivo]
        columnas = ['—'] + df.columns.tolist()
        
        selector_x.options = columnas
        selector_y.options = columnas
        selector_color.options = columnas
        
        # Seleccionar primeros valores por defecto
        if len(columnas) > 1:
            selector_x.value = columnas[1]
            selector_y.value = columnas[2] if len(columnas) > 2 else columnas[1]
            selector_color.value = '—'
        
        label_estado.value = f"<b style='color:#0D9488;'>✓ Archivo '{archivo}' cargado ({df.shape[0]} filas × {df.shape[1]} cols)</b>"


# ── Función para generar gráfico ───────────────────────────────────────────
def generar_grafico(button):
    """Genera y muestra el gráfico solicitado."""
    with output_grafico:
        output_grafico.clear_output(wait=True)
        
        archivo = selector_archivo.value
        eje_x = selector_x.value
        eje_y = selector_y.value
        tipo = selector_tipo.value
        color = selector_color.value
        
        # Validar selecciones
        if archivo not in DATOS or eje_x == '—' or (eje_y == '—' and tipo != 'histograma'):
            label_estado.value = "<b style='color:#DC2626;'>✗ Seleccione archivo, eje X y eje Y válidos</b>"
            print("⚠️  Parámetros inválidos")
            return
        
        try:
            df = DATOS[archivo]
            label_estado.value = "<b style='color:#EA580C;'>⏳ Generando gráfico...</b>"
            
            # Generar gráfico
            widget_img = grafico_automatico(df, eje_x, eje_y, tipo, 
                                           color if color != '—' else None)
            display(widget_img)
            
            label_estado.value = f"<b style='color:#0D9488;'>✓ Gráfico generado: {tipo.upper()}</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error: {str(e)}</b>"


# ── Función para mostrar estadísticas ───────────────────────────────────────
def mostrar_estadisticas(button):
    """Muestra el resumen estadístico del archivo seleccionado."""
    with output_estadisticas:
        output_estadisticas.clear_output(wait=True)
        
        archivo = selector_archivo.value
        
        if archivo not in DATOS:
            print("⚠️  Seleccione un archivo válido")
            return
        
        try:
            df = DATOS[archivo]
            label_estado.value = "<b style='color:#EA580C;'>⏳ Calculando estadísticas...</b>"
            
            html_resumen = resumen_estadistico_html(df)
            from IPython.display import HTML
            display(HTML(html_resumen))
            
            label_estado.value = f"<b style='color:#0D9488;'>✓ Estadísticas de '{archivo}'</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error: {str(e)}</b>"


# ── Conectar eventos ────────────────────────────────────────────────────────
selector_archivo.observe(actualizar_columnas, names='value')
btn_generar.on_click(generar_grafico)
btn_estadisticas.on_click(mostrar_estadisticas)
btn_recargar.on_click(recargar_archivos)


# ╔══════════════════════════════════════════════════════════╗
# ║  INTERFAZ VISUAL DEL DASHBOARD                          ║
# ╚══════════════════════════════════════════════════════════╝

# Sección: CONTROLES
seccion_controles = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>⚙️ CONTROLES</h3>"),
    widgets.HBox([selector_archivo], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_x], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_y], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_tipo], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_color], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([btn_generar, btn_estadisticas], layout=widgets.Layout(gap='10px', margin='15px 0px')),
    widgets.HBox([btn_recargar], layout=widgets.Layout(margin='10px 0px')),
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    width='320px'
))

# Sección: GRÁFICO PRINCIPAL
seccion_grafico = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>📊 GRÁFICO</h3>"),
    output_grafico
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    flex='1'
))

# Sección: ESTADÍSTICAS
seccion_estadisticas = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>📈 RESUMEN ESTADÍSTICO</h3>"),
    output_estadisticas
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    flex='1'
))

# Sección: ESTADO
seccion_estado = widgets.VBox([
    label_estado
], layout=widgets.Layout(
    border='1px solid #D1D5DB',
    padding='12px',
    margin='10px 0px',
    background_color='#F9FAFB'
))

# ╔══════════════════════════════════════════════════════════╗
# ║  LAYOUT PRINCIPAL                                       ║
# ╚══════════════════════════════════════════════════════════╝

# Título principal
titulo = widgets.HTML("""
<div style='text-align:center; padding:20px 0px; background:linear-gradient(135deg, #2563EB, #0D9488); 
            color:white; border-radius:8px; margin-bottom:20px;'>
    <h1 style='margin:0; font-size:28px;'>📊 DASHBOARD INTERACTIVO</h1>
    <p style='margin:5px 0 0 0; font-size:14px; opacity:0.9;'>
        Análisis y Visualización de Datos — Acceso y Rendimiento Universitario en Chile
    </p>
</div>
""")

# Layout principal: controles a la izquierda, gráficos a la derecha
layout_principal = widgets.HBox([
    seccion_controles,
    widgets.VBox([seccion_grafico, seccion_estadisticas])
], layout=widgets.Layout(
    gap='15px'
))

# Contenedor final
dashboard_final = widgets.VBox([
    titulo,
    layout_principal,
    seccion_estado
], layout=widgets.Layout(
    width='100%',
    padding='20px'
))

# ── Mostrar el dashboard ────────────────────────────────────────────────────
display(dashboard_final)

# Inicializar con el primer archivo
if DATOS:
    primer_archivo = list(DATOS.keys())[0]
    selector_archivo.value = primer_archivo
    actualizar_columnas({'new': primer_archivo})

print("Dashboard interactivo cargado ✓  —  Interfaz lista para usar")


Dashboard interactivo cargado ✓  —  Interfaz lista para usar


In [8]:
# %% [markdown]
# #### Celda 3: Lógica del Motor Reactivo (Procesamiento de Callbacks y Gráficos en Vivo)

# %%
ANCHO_FIJO = '980px'
ALTO_FIJO = '580px'

# 1. Enrutador central superior
def alternar_pestanas(change):
    with contenedor_cuerpo:
        clear_output(wait=True)
        pestana_activa = change['new'] if change else tabs_navegacion.value
        if pestana_activa == 'KPIs':
            display(layout_tab_kpis)
            actualizar_kpi_cards(None)
        elif pestana_activa == 'GRÁFICOS':
            display(layout_tab_graficos)
            actualizar_imagen_grafico(None)
        elif pestana_activa == 'MAPAS':
            display(layout_tab_mapas)
            actualizar_imagen_mapa(None)
        elif pestana_activa == 'ML (RANDOM FOREST PESOS)':
            display(layout_tab_ml)

tabs_navegacion.observe(alternar_pestanas, names='value')

# 2. Callback del Panel de KPIs (Operaciones vectoriales rápidas sobre Pandas)
def actualizar_kpi_cards(change):
    with area_kpi_cards:
        clear_output(wait=True)
        inst = selector_kpi_institucion.value
        carr = selector_kpi_carrera.value
        
        if inst == '---' or carr == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'><h4>[ Selecciona una institución y una carrera para desplegar las métricas KPI ]</h4></div>"))
        else:
            df_filtro = df_ing[(df_ing[col_inst] == inst) & (df_ing['Carrera Genérica'] == carr)]
            
            if not df_filtro.empty:
                df_reciente = df_filtro[df_filtro['Año'] == df_filtro['Año'].max()]
                vacantes = df_reciente['Vacantes'].sum()
                matricula_1er = df_reciente['Matrícula Primer Año'].sum()
                v_llenado = f"{(matricula_1er / vacantes * 100):.1f}%" if vacantes > 0 else "100.0%"
                
                mujeres_1er = df_reciente['Matrícula primer año mujeres'].sum()
                v_fem = f"{(mujeres_1er / matricula_1er * 100):.1f}%" if matricula_1er > 0 else "0.0%"
                v_total = f"{int(df_reciente['Matrícula Total'].sum()):,}".replace(',', '.')
            else:
                v_llenado, v_fem, v_total = "--%", "--%", "--"
                
            html_content = f"""
            <div style='font-family: sans-serif; padding: 5px; height:100%;'>
                <h4 style='color: #2c3e50; margin-top: 0; margin-bottom: 5px;'>Análisis de Rendimiento: {inst}</h4>
                <h5 style='color: #555; margin-top: 0; margin-bottom: 20px; font-weight: normal;'>Programa: <b>{carr}</b></h5>
                <div style='display: flex; justify-content: space-between; gap: 10px; margin-bottom: 20px;'>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5bc0de; padding: 12px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.8em; color: #777; font-weight: bold; text-transform: uppercase; margin-bottom: 5px;'>Ocupación</div>
                        <div style='font-size: 1.6em; font-weight: bold; color: #333;'>{v_llenado}</div>
                    </div>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5cb85c; padding: 12px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.8em; color: #777; font-weight: bold; text-transform: uppercase; margin-bottom: 5px;'>Mix de Género</div>
                        <div style='font-size: 1.6em; font-weight: bold; color: #333;'>{v_fem}</div>
                    </div>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #f0ad4e; padding: 12px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.8em; color: #777; font-weight: bold; text-transform: uppercase; margin-bottom: 5px;'>Comunidad Activa</div>
                        <div style='font-size: 1.6em; font-weight: bold; color: #333;'>{v_total}</div>
                    </div>
                </div>
                <div style='background-color: #fff; border: 1px solid #e3e3e3; padding: 15px; border-radius: 4px;'>
                    <h5 style='margin-top: 0; color: #333; font-size: 1em;'>Resumen Consolidado</h5>
                    <p style='font-size: 0.88em; color: #555; line-height: 1.4; margin-bottom: 0;'>Los indicadores clave se calculan dinámicamente consultando la memoria del DataFrame. Cambiar cualquier filtro recalculará los valores de forma inmediata.</p>
                </div>
            </div>
            """
            display(widgets.HTML(html_content))

selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')

# 3. Callback del Panel de Gráficos (Renderizado en memoria viva)
def actualizar_imagen_grafico(change):
    with area_imagen_grafico:
        clear_output(wait=True)
        opcion = selector_graficos.value
        carrera_dim = selector_dimensiones.value
        
        if opcion == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'><h4>[ Selecciona un reporte para generar el gráfico dinámico ]</h4></div>"))
        else:
            df_plot = df_ing if carrera_dim == '---' else df_ing[df_ing['Carrera Genérica'] == carrera_dim]
            titulo_contexto = "Global Ingeniería" if carrera_dim == '---' else carrera_dim
            
            fig, ax = plt.subplots(figsize=(6.5, 4))
            if opcion == 'Evolución de Arancel Promedio':
                data = df_plot.groupby('Año')['Valor de arancel'].median().reset_index()
                sns.lineplot(data=data, x='Año', y='Valor de arancel', marker='D', color='#2ecc71', linewidth=2.5, ax=ax)
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x*1e-6:.1f}M"))
                ax.set_ylabel('Arancel Promedio')
            elif opcion == 'Evolución de Matrícula Total':
                data = df_plot.groupby('Año')['Matrícula Total'].sum().reset_index()
                sns.lineplot(data=data, x='Año', y='Matrícula Total', marker='s', color='#54a0ff', linewidth=2.5, ax=ax)
                ax.set_ylabel('Cantidad de Alumnos')
            elif opcion == 'Tendencia de Ingreso Femenino':
                data = df_plot.groupby('Año')[['Matrícula primer año hombres', 'Matrícula primer año mujeres']].sum().reset_index()
                data['Pct_Mujeres'] = (data['Matrícula primer año mujeres'] / (data['Matrícula primer año hombres'] + data['Matrícula primer año mujeres'])) * 100
                sns.lineplot(data=data, x='Año', y='Pct_Mujeres', marker='o', color='#ff7597', linewidth=2.5, ax=ax)
                ax.set_ylabel('Porcentaje de Mujeres (%)')
                ax.set_ylim(0, 60)
            
            ax.set_title(f'{opcion}\n({titulo_contexto})', fontweight='bold', fontsize=11)
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
            ax.set_xlabel('Año Académico')
            plt.tight_layout()
            plt.show()

selector_graficos.observe(actualizar_imagen_grafico, names='value')
selector_dimensiones.observe(actualizar_imagen_grafico, names='value')

# 4. Callback del Panel de Mapas (Distribución Territorial)
def actualizar_imagen_mapa(change):
    with area_imagen_mapa:
        clear_output(wait=True)
        opcion = selector_mapas.value
        
        if opcion == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'><h4>[ Elige una variable territorial para calcular el reporte regional ]</h4></div>"))
        else:
            df_mapa = df_ing[df_ing['Año'] == df_ing['Año'].max()]
            fig, ax = plt.subplots(figsize=(6.5, 4))
            
            if col_reg in df_mapa.columns:
                if opcion == 'Distribución de Alumnos por Región/Sede':
                    data = df_mapa.groupby(col_reg)['Matrícula Total'].sum().sort_values(ascending=False).head(8).reset_index()
                    sns.barplot(data=data, x='Matrícula Total', y=col_reg, hue=col_reg, palette='viridis', legend=False, ax=ax)
                    ax.set_xlabel('Alumnos Activos')
                elif opcion == 'Comparativa de Aranceles Regiones':
                    data = df_mapa.groupby(col_reg)['Valor de arancel'].median().sort_values(ascending=False).head(8).reset_index()
                    sns.barplot(data=data, x='Valor de arancel', y=col_reg, hue=col_reg, palette='coolwarm', legend=False, ax=ax)
                    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x*1e-6:.1f}M"))
                    ax.set_xlabel('Valor Arancel Anual')
                
                ax.set_title(opcion, fontweight='bold', fontsize=11)
                ax.set_ylabel('')
                plt.tight_layout()
                plt.show()
            else:
                display(widgets.HTML("<div style='color:red; padding:20px;'>⚠️ Error: Columna territorial no detectada.</div>"))

selector_mapas.observe(actualizar_imagen_mapa, names='value')

# 5. Configuración de Infraestructura de Acceso Seguro
txt_usuario = widgets.Text(description='Usuario:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
txt_password = widgets.Password(description='Clave:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
btn_login = widgets.Button(description='Autenticar', button_style='primary', icon='lock', layout=widgets.Layout(margin='20px 0px 5px 0px', width='280px'))
html_feedback = widgets.HTML(value="")

formulario_interno = widgets.VBox([widgets.HTML("<h3 style='text-align: center; font-family: sans-serif; color: #333; margin-top:0;'>SISTEMA DE ACCESO</h3><hr style='width: 100%; border: 0; border-top: 1px solid #ccc;'>"), txt_usuario, txt_password, btn_login, html_feedback], layout=widgets.Layout(width='360px', padding='25px', border='1px solid #ccc', bg_color='#ffffff', align_items='center', border_radius='4px'))
cuadro_login = widgets.VBox([formulario_interno], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', justify_content='center', align_items='center'))
dashboard_final = widgets.VBox([estilos_css, tabs_navegacion, linea_separadora, contenedor_cuerpo], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', padding='20px'))

def validar_credenciales(b):
    if txt_usuario.value == 'admin' and txt_password.value == 'admin':
        with lienzo_maestro:
            clear_output()
            display(dashboard_final)
            alternar_pestanas(None)
    else:
        txt_password.value = ""
        html_feedback.value = "<div style='color: #d9534f; font-weight: bold; text-align: center; margin-top: 12px; font-family: sans-serif;'>Error: Credenciales Incorrectas</div>"

btn_login.on_click(validar_credenciales)
lienzo_maestro = widgets.Output()
print("✅ CELDA 3 CONTINUA: Motores de callback reactivos configurados.")

✅ CELDA 3 CONTINUA: Motores de callback reactivos configurados.


In [9]:
# %% [markdown]
# #### Celda 4: Despliegue de la Aplicación Base en Pantalla

# %%
# Desplegamos el nodo de salida raíz del lienzo maestro
display(lienzo_maestro)

# Forzamos a que pinte el bloque de Login en el arranque
with lienzo_maestro:
    clear_output()
    display(cuadro_login)

Output()